In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, PayloadSchemaType, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import openai
import fastembed


/home/nurda/ai-engineering/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  
)

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
def get_embedding(text, model="nomic-embed-text"):
    safe_text = text[:10000]
    response = client.embeddings.create(
        input=safe_text,
        model=model
    )
    return response.data[0].embedding

In [5]:
def retrieve_data(query, qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon_items_collection1_hybrid_search",
        prefetch = [
            Prefetch(
             query=query_embedding,
             using="nomic-embed-text",
             limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm-25",
                limit=20
                )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=top_k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []

    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_contexts.append(point.payload["description"])
        retrieved_context_ratings.append(point.payload["average_rating"])
        similarity_scores.append(point.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [6]:
response_data = retrieve_data("Can i get some tablet?",qdrant_client=qdrant_client, top_k=10)

In [7]:
response_data

{'retrieved_context_ids': ['B0B1X471HB',
  'B09XX9YTBP',
  'B0B1X471HB',
  'B09XX9YTBP',
  'B09Q82NZG9',
  'B0C8SXP4P2',
  'B0C8SXP4P2',
  'B09Q82NZG9',
  'B09XDWNL5K',
  'B09V5CSFJV'],
 'retrieved_contexts': ['TJD Android Tablet 7.9 inch, Android 11 Tablets, 2048x1536 IPS Display, 2GB RAM 32GB ROM(512GB Expandable), Quad-Core Processor, 8MP Camera/Google Certified/Dual Speakers/Bluetooth/2.4GHz WiFi/Type-C 2K DISPLAY - 7.9-inch size tablet display featured with the real 2K(2048*1536) resolution. The IPS touch screen allows you to easily set up and when you are watching movies and images with more clearly view and brighter color. DOUBLE GLASS SCREEN - The tablet touch screen is made of G+G technology, which can support up to ten points of touch while maintaining this sensitive touch smoothly. CPU+BATTERY - Tablet with 1.6hz Quad-Core High-Performance CPU and This Android 11 tablet come with a long-lasting battery that can last longer. Ideal for long commutes and travel, reading, watchi

## Reranker

In [8]:
import transformers.utils.import_utils

transformers.utils.import_utils.is_torch_fx_available = lambda: True

from FlagEmbedding import FlagReranker

query = "Can i get some tablet?" 

reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

pairs = [[query, context] for context in response_data['retrieved_contexts']]
new_scores = reranker.compute_score(pairs)

combined_results = list(zip(
    response_data['retrieved_context_ids'],
    response_data['retrieved_contexts'],
    response_data['similarity_scores'],  
    new_scores                           
))

combined_results.sort(key=lambda x: x[3], reverse=True)

reranked_data = {
    'retrieved_context_ids': [item[0] for item in combined_results],
    'retrieved_contexts': [item[1] for item in combined_results],
    'old_similarity_scores': [item[2] for item in combined_results],
    'reranker_scores': [item[3] for item in combined_results]
}

print(f"Лучший документ ID: {reranked_data['retrieved_context_ids'][0]}")
print(f"Новый скор: {reranked_data['reranker_scores'][0]}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
/home/nurda/ai-engineering/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a cal

Лучший документ ID: B09V5CSFJV
Новый скор: -3.1895508766174316


In [9]:
reranked_data

{'retrieved_context_ids': ['B09V5CSFJV',
  'B0C8SXP4P2',
  'B0C8SXP4P2',
  'B09XDWNL5K',
  'B0B1X471HB',
  'B0B1X471HB',
  'B09XX9YTBP',
  'B09XX9YTBP',
  'B09Q82NZG9',
  'B09Q82NZG9'],
 'retrieved_contexts': ['2023 Newest Android 12 Tablet 10 inch with Keyboard Case Bundle,4GB+64GB ROM/512GB Gaming Tablets,2MP+8MP Dual Cameras,Quad Core Processor,WiFi,Bluetooth,1280x800 IPS HD Display,Google Certified 👍【4G ROM & 64GB RAM Storage】This Android tablet has a high-performance quad-core processor with 64GB large-capacity storage,Expandable external storage 256G, whether you are playing games, watching movies or online processing work files are very smooth, no longer need to worry about the lack of memory to affect the experience 👍【Android 12 OS】The latest version of Android 12 system, the industry standard for high performance graphics. Reveals tons of new adventures to your tablet. Easy multi-tasking, rich notifications, resizable widgets and deep interactivity. The device will allow you t